In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving blinkit_customer_feedback.csv to blinkit_customer_feedback.csv
Saving blinkit_products.csv to blinkit_products.csv
Saving blinkit_orders.csv to blinkit_orders.csv
Saving blinkit_order_items.csv to blinkit_order_items.csv
Saving blinkit_marketing_performance.csv to blinkit_marketing_performance.csv
Saving blinkit_inventoryNew.csv to blinkit_inventoryNew.csv
Saving blinkit_inventory.csv to blinkit_inventory.csv
Saving blinkit_delivery_performance.csv to blinkit_delivery_performance.csv
Saving blinkit_customers.csv to blinkit_customers.csv


In [3]:
products = pd.read_csv('blinkit_products.csv')
orders = pd.read_csv('blinkit_orders.csv')
order_items = pd.read_csv('blinkit_order_items.csv')
customers = pd.read_csv('blinkit_customers.csv')
delivery = pd.read_csv('blinkit_delivery_performance.csv')
marketing = pd.read_csv('blinkit_marketing_performance.csv')
inventory = pd.read_csv('blinkit_inventory.csv')
inventory_new = pd.read_csv('blinkit_inventoryNew.csv')
feedback = pd.read_csv('blinkit_customer_feedback.csv')

In [4]:
products.isnull().sum()
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_date,0
promised_delivery_time,0
actual_delivery_time,0
delivery_status,0
order_total,0
payment_method,0
delivery_partner_id,0
store_id,0


In [5]:
for name, df in {
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "delivery": delivery,
    "marketing": marketing,
    "inventory": inventory,
    "inventory_new": inventory_new,
    "feedback": feedback
}.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    print(df.info())
    print(df.head())


PRODUCTS
(268, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268 entries, 0 to 267
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         268 non-null    int64  
 1   product_name       268 non-null    object 
 2   category           268 non-null    object 
 3   brand              268 non-null    object 
 4   price              268 non-null    float64
 5   mrp                268 non-null    float64
 6   margin_percentage  268 non-null    float64
 7   shelf_life_days    268 non-null    int64  
 8   min_stock_level    268 non-null    int64  
 9   max_stock_level    268 non-null    int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 21.1+ KB
None
   product_id product_name             category                    brand  \
0      153019       Onions  Fruits & Vegetables               Aurora LLC   
1       11422     Potatoes  Fruits & Vegetables           Ramaswamy-Tata   
2  

# Standardize column names

In [6]:
dfs = [products, orders, order_items, customers, delivery, marketing, inventory, inventory_new, feedback]

for df in dfs:
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

Remove duplicates

In [7]:
for df in dfs:
    df.drop_duplicates(inplace=True)

In [8]:
for name, df in zip(
    ["products","orders","order_items","customers","delivery","marketing","inventory","inventory_new","feedback"],
    dfs):
    print(name)
    print(df.isnull().sum())
    print("-"*40)

products
product_id           0
product_name         0
category             0
brand                0
price                0
mrp                  0
margin_percentage    0
shelf_life_days      0
min_stock_level      0
max_stock_level      0
dtype: int64
----------------------------------------
orders
order_id                  0
customer_id               0
order_date                0
promised_delivery_time    0
actual_delivery_time      0
delivery_status           0
order_total               0
payment_method            0
delivery_partner_id       0
store_id                  0
dtype: int64
----------------------------------------
order_items
order_id      0
product_id    0
quantity      0
unit_price    0
dtype: int64
----------------------------------------
customers
customer_id          0
customer_name        0
email                0
phone                0
address              0
area                 0
pincode              0
registration_date    0
customer_segment     0
total_orders       

In [9]:
delivery['reasons_if_delayed'] = delivery['reasons_if_delayed'].fillna('On Time')

In [10]:
delivery['promised_time'] = pd.to_datetime(delivery['promised_time'])
delivery['actual_time'] = pd.to_datetime(delivery['actual_time'])

delivery['is_delayed'] = delivery['actual_time'] > delivery['promised_time']

delivery['delay_minutes'] = (delivery['actual_time'] - delivery['promised_time']).dt.total_seconds() / 60
delivery['delay_minutes'] = delivery['delay_minutes'].clip(lower=0)

In [11]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

orders['hour'] = orders['order_date'].dt.hour
orders['day'] = orders['order_date'].dt.day
orders['month'] = orders['order_date'].dt.month
orders['weekday'] = orders['order_date'].dt.day_name()

In [12]:
order_items['total_price'] = order_items['quantity'] * order_items['unit_price']

In [13]:
customers['total_orders'] = customers['total_orders'].fillna(0)

In [14]:
df = orders.merge(order_items, on='order_id', how='left') \
           .merge(products, on='product_id', how='left') \
           .merge(customers, on='customer_id', how='left') \
           .merge(delivery, on='order_id', how='left') \
           .merge(feedback, on='order_id', how='left')

In [15]:
df.shape
df.isnull().sum()

,0
order_id,0
customer_id_x,0
order_date,0
promised_delivery_time,0
actual_delivery_time,0
delivery_status_x,0
order_total,0
payment_method,0
delivery_partner_id_x,0
store_id,0


ADVANCED FEATURES:
Delivery Efficiency,
Peak Hours,
Order Size

In [16]:
df['promised_duration_minutes'] = (df['promised_time'] - df['order_date']).dt.total_seconds() / 60
df['delivery_efficiency'] = df['promised_duration_minutes'] / df['delivery_time_minutes']

In [17]:
order_size = df.groupby('order_id')['quantity'].sum().reset_index()
order_size.columns = ['order_id', 'order_size']

df = df.merge(order_size, on='order_id', how='left')

In [18]:
print("Total Orders:", df['order_id'].nunique())
print("Total Revenue:", df['total_price'].sum())
print("Avg Delivery Time:", df['actual_time'].mean())
print("Delay %:", df['is_delayed'].mean() * 100)
print("Avg Rating:", df['rating'].mean())

Total Orders: 5000
Total Revenue: 4972415.43
Avg Delivery Time: 2024-01-09 00:55:32.323200
Delay %: 61.96
Avg Rating: 3.3444


In [24]:
df['is_peak'] = df['hour'].apply(lambda x: True if (8 <= x <= 11) or (18 <= x <= 22) else False)

In [27]:
# Peak vs Non-peak delay
df.groupby('is_peak')['delay_minutes'].mean()

# Store performance
df.groupby('store_id')['is_delayed'].mean()

# Delivery partner performance
df.groupby('delivery_partner_id')['actual_duration_minutes'].mean()

# Delay vs rating
df.groupby('is_delayed')['rating'].mean()

KeyError: 'Column not found: actual_duration_minutes'

In [20]:
df.rename(columns={
    'customer_id_x': 'customer_id',
    'delivery_partner_id_x': 'delivery_partner_id'
}, inplace=True)

df.drop(columns=['customer_id_y', 'delivery_partner_id_y'], errors='ignore', inplace=True)

In [21]:
df.rename(columns={
    'promised_delivery_time': 'promised_time',
    'actual_delivery_time': 'actual_time'
}, inplace=True)

In [28]:
df['promised_duration_minutes'] = (
    df['promised_time'] - df['order_date']
).dt.total_seconds() / 60

df['actual_duration_minutes'] = (
    df['actual_time'] - df['order_date']
).dt.total_seconds() / 60

ValueError: cannot reindex on an axis with duplicate labels

In [29]:
df.columns[df.columns.duplicated()]

Index(['promised_time', 'actual_time'], dtype='object')

In [30]:
df[['order_date', 'promised_time', 'actual_time']].dtypes

,0
order_date,datetime64[ns]
promised_time,object
promised_time,datetime64[ns]
actual_time,object
actual_time,datetime64[ns]


In [31]:
df = df.loc[:, ~df.columns.duplicated(keep='last')]

In [32]:
df[['order_date','promised_time','actual_time']].dtypes

,0
order_date,datetime64[ns]
promised_time,datetime64[ns]
actual_time,datetime64[ns]


In [33]:
df['promised_duration_minutes'] = (
    df['promised_time'] - df['order_date']
).dt.total_seconds() / 60

df['actual_duration_minutes'] = (
    df['actual_time'] - df['order_date']
).dt.total_seconds() / 60

/tmp/ipykernel_1140/2588290160.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['promised_duration_minutes'] = (
/tmp/ipykernel_1140/2588290160.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['actual_duration_minutes'] = (


In [34]:
df['delivery_efficiency'] = (
    df['promised_duration_minutes'] / df['actual_duration_minutes']
)

/tmp/ipykernel_1140/2365782354.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delivery_efficiency'] = (


In [35]:
df['delivery_efficiency'] = df['delivery_efficiency'].replace([np.inf, -np.inf], np.nan)
df['delivery_efficiency'] = df['delivery_efficiency'].fillna(0)

/tmp/ipykernel_1140/2867490394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delivery_efficiency'] = df['delivery_efficiency'].replace([np.inf, -np.inf], np.nan)
/tmp/ipykernel_1140/2867490394.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delivery_efficiency'] = df['delivery_efficiency'].fillna(0)


Delay minutes (direct business feature)

In [36]:
df.describe()

,order_id,customer_id,order_date,order_total,delivery_partner_id,store_id,hour,day,month,product_id,...,actual_time,delivery_time_minutes,distance_km,delay_minutes,feedback_id,rating,promised_duration_minutes,delivery_efficiency,order_size,actual_duration_minutes
count,5.000000e+03,5.000000e+03,5000,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,...,5000,5000.000000,5000.000000,5000.000000,5.000000e+03,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000
mean,5.029129e+09,5.009685e+07,2024-01-09 00:36:06.259200,2201.86170,50050.318200,4999.689000,11.522600,15.778400,6.643600,509974.939600,...,2024-01-09 00:55:32.323200,4.443000,2.718048,5.381800,5.013616e+06,3.34440,14.991400,0.894972,2.006800,19.434400
min,6.046500e+04,3.181300e+04,2023-03-16 08:10:44,13.25000,43.000000,1.000000,0.000000,1.000000,1.000000,4452.000000,...,2023-03-16 08:24:44,-5.000000,0.500000,0.000000,9.470000e+02,1.00000,10.000000,0.250000,1.000000,5.000000
25%,2.531421e+09,2.404314e+07,2023-08-17 01:54:09.249999872,1086.21500,24928.500000,2509.250000,5.750000,8.000000,4.000000,257719.000000,...,2023-08-17 02:27:24.249999872,-1.000000,1.590000,0.000000,2.576690e+06,3.00000,12.000000,0.647059,1.000000,13.000000
50%,5.074378e+09,4.997808e+07,2024-01-07 11:35:32,2100.69000,50262.500000,4987.000000,12.000000,16.000000,7.000000,540618.000000,...,2024-01-07 11:51:02,2.000000,2.690000,2.000000,5.005833e+06,4.00000,15.000000,0.857143,2.000000,18.000000
75%,7.488579e+09,7.621215e+07,2024-06-03 20:53:15.750000128,3156.88250,74478.250000,7500.750000,18.000000,23.000000,9.000000,747801.000000,...,2024-06-03 21:09:30.750000128,8.000000,3.850000,8.000000,7.486478e+06,4.00000,18.000000,1.111111,3.000000,24.000000
max,9.998298e+09,9.989390e+07,2024-11-04 20:29:15,6721.46000,99968.000000,9995.000000,23.000000,31.000000,12.000000,993331.000000,...,2024-11-04 20:47:15,30.000000,5.000000,30.000000,9.999293e+06,5.00000,20.000000,2.000000,3.000000,50.000000
std,2.863533e+09,2.919082e+07,NaN,1303.02438,28802.276922,2886.089242,6.945901,8.784892,3.058701,293678.307475,...,NaN,8.063929,1.290306,7.234756,2.857341e+06,1.18982,3.160336,0.336981,0.820542,8.659486


In [37]:
df.drop(columns=['delivery_time_minutes'], inplace=True)

/tmp/ipykernel_1140/546184003.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['delivery_time_minutes'], inplace=True)


In [38]:
df['delay_minutes'] = (
    df['actual_duration_minutes'] - df['promised_duration_minutes']
).clip(lower=0)

/tmp/ipykernel_1140/1020423973.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delay_minutes'] = (


ADVANCED FEATURE:
Delay Category

In [39]:
df['delay_category'] = pd.cut(
    df['delay_minutes'],
    bins=[-1, 5, 15, 100],
    labels=['Low Delay', 'Medium Delay', 'High Delay']
)

/tmp/ipykernel_1140/3366609064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delay_category'] = pd.cut(


In [40]:
df.isnull().sum()
df.describe()

,order_id,customer_id,order_date,order_total,delivery_partner_id,store_id,hour,day,month,product_id,...,promised_time,actual_time,distance_km,delay_minutes,feedback_id,rating,promised_duration_minutes,delivery_efficiency,order_size,actual_duration_minutes
count,5.000000e+03,5.000000e+03,5000,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,...,5000,5000,5000.000000,5000.000000,5.000000e+03,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000
mean,5.029129e+09,5.009685e+07,2024-01-09 00:36:06.259200,2201.86170,50050.318200,4999.689000,11.522600,15.778400,6.643600,509974.939600,...,2024-01-09 00:51:05.743200256,2024-01-09 00:55:32.323200,2.718048,5.381800,5.013616e+06,3.34440,14.991400,0.894972,2.006800,19.434400
min,6.046500e+04,3.181300e+04,2023-03-16 08:10:44,13.25000,43.000000,1.000000,0.000000,1.000000,1.000000,4452.000000,...,2023-03-16 08:27:44,2023-03-16 08:24:44,0.500000,0.000000,9.470000e+02,1.00000,10.000000,0.250000,1.000000,5.000000
25%,2.531421e+09,2.404314e+07,2023-08-17 01:54:09.249999872,1086.21500,24928.500000,2509.250000,5.750000,8.000000,4.000000,257719.000000,...,2023-08-17 02:13:24.249999872,2023-08-17 02:27:24.249999872,1.590000,0.000000,2.576690e+06,3.00000,12.000000,0.647059,1.000000,13.000000
50%,5.074378e+09,4.997808e+07,2024-01-07 11:35:32,2100.69000,50262.500000,4987.000000,12.000000,16.000000,7.000000,540618.000000,...,2024-01-07 11:47:02,2024-01-07 11:51:02,2.690000,2.000000,5.005833e+06,4.00000,15.000000,0.857143,2.000000,18.000000
75%,7.488579e+09,7.621215e+07,2024-06-03 20:53:15.750000128,3156.88250,74478.250000,7500.750000,18.000000,23.000000,9.000000,747801.000000,...,2024-06-03 21:09:30.750000128,2024-06-03 21:09:30.750000128,3.850000,8.000000,7.486478e+06,4.00000,18.000000,1.111111,3.000000,24.000000
max,9.998298e+09,9.989390e+07,2024-11-04 20:29:15,6721.46000,99968.000000,9995.000000,23.000000,31.000000,12.000000,993331.000000,...,2024-11-04 20:43:15,2024-11-04 20:47:15,5.000000,30.000000,9.999293e+06,5.00000,20.000000,2.000000,3.000000,50.000000
std,2.863533e+09,2.919082e+07,NaN,1303.02438,28802.276922,2886.089242,6.945901,8.784892,3.058701,293678.307475,...,NaN,NaN,1.290306,7.234756,2.857341e+06,1.18982,3.160336,0.336981,0.820542,8.659486


In [41]:
df['delay_minutes'] = df['delay_minutes'].clip(lower=0)

/tmp/ipykernel_1140/585402730.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delay_minutes'] = df['delay_minutes'].clip(lower=0)


In [42]:
(df['actual_duration_minutes'] < df['promised_duration_minutes']).sum()

np.int64(1563)

In [43]:
df['delivery_efficiency'].describe()



,delivery_efficiency
count,5000.000000
mean,0.894972
std,0.336981
min,0.250000
25%,0.647059
50%,0.857143
75%,1.111111
max,2.000000


In [44]:
df.drop(columns=['delivery_time_minutes'], inplace=True, errors='ignore')

/tmp/ipykernel_1140/2072846277.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['delivery_time_minutes'], inplace=True, errors='ignore')


In [45]:
df['delay_minutes'] = (
    df['actual_duration_minutes'] - df['promised_duration_minutes']
).clip(lower=0)

/tmp/ipykernel_1140/1020423973.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['delay_minutes'] = (


In [46]:
df = df[df['delivery_efficiency'] >= 0]
df = df[df['delivery_efficiency'] <= 3]

In [47]:
df = df.drop_duplicates()

In [48]:
df['delivery_status'] = df['delivery_efficiency'].apply(
    lambda x: 'On Time / Early' if x >= 1 else 'Delayed'
)

In [49]:
df['delay_bucket'] = pd.cut(
    df['delay_minutes'],
    bins=[-1,5,15,100],
    labels=['0-5 min','5-15 min','15+ min']
)

In [50]:
df['order_value_bucket'] = pd.cut(
    df['order_total'],
    bins=[0,1000,3000,7000],
    labels=['Low','Medium','High']
)

In [51]:
df.drop(['delivery_status_x', 'delivery_status_y'], axis=1, inplace=True)

In [52]:
date_cols = ['order_date', 'registration_date', 'feedback_date']
time_cols = ['promised_time', 'actual_time']

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

for col in time_cols:
    df[col] = pd.to_datetime(df[col])

In [53]:
df['phone'] = df['phone'].astype(str)

In [54]:
df.drop(['price'], axis=1, inplace=True)

In [55]:
df['calc_total'] = df['quantity'] * df['unit_price']
(df['calc_total'] - df['total_price']).abs().sum()

np.float64(0.0)

In [56]:
df[df['delay_minutes'] < 0]

,order_id,customer_id,order_date,order_total,payment_method,delivery_partner_id,store_id,hour,day,month,...,promised_duration_minutes,delivery_efficiency,order_size,is_peak,actual_duration_minutes,delay_category,delivery_status,delay_bucket,order_value_bucket,calc_total


In [57]:
df['calc_duration'] = df['actual_duration_minutes'] - df['promised_duration_minutes']

In [122]:
df.dtypes

,0
order_id,int64
customer_id,int64
order_date,datetime64[ns]
order_total,float64
payment_method,object
delivery_partner_id,int64
store_id,int64
hour,int32
day,int32
month,int32


In [58]:
(df['delay_minutes'] < 0).sum()

np.int64(0)

In [59]:
(df['quantity'] * df['unit_price'] - df['total_price']).abs().sum()

np.float64(0.0)

In [60]:
df['rating'].unique()

array([4, 3, 2, 5, 1])

In [61]:
df['delivery_status'].unique()
df['category'].unique()
df['sentiment'].unique()

array(['Neutral', 'Negative', 'Positive'], dtype=object)

In [62]:
df['is_peak'].unique()

array([ True, False])

In [63]:
df['is_peak'].value_counts(dropna=False)

,count
is_peak,
False,3118
True,1882


In [64]:
df['is_peak'] = df['hour'].apply(lambda x: True if (8 <= x <= 11) or (18 <= x <= 22) else False)

In [65]:
df['is_peak'].unique()

array([ True, False])

In [66]:
df['delivery_status'] = df['delivery_status'].str.strip().str.title()
df['category'] = df['category'].str.strip().str.title()

In [67]:
df.shape

(5000, 56)

In [68]:
df.to_csv('FINAL(2)_CLEANED_DASHBOARD_DATA.csv', index=False)